# 05 — Preference filtering + travel time

Demonstrates `reasoning/preference_filter.py`: "find me a <POI type>, with
<amenities>, ranked by how fast I can actually get there" — combining the
category/amenity data from KG Modelling with the GTFS travel-time reasoning.

**Updated for the 2-transfer routing extension** (see
`docs/reasoning_layer_decisions.md`): `find_pois()` now uses
`router.reachable_from()` once per query plus `router.travel_time_to()` per
candidate, instead of calling `estimate_travel_time()` per candidate — the
difference between ~3 seconds and several minutes for a 1,000-POI category.
Results are also far less sparse now that direct-only's <1% connectivity rate
no longer applies.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from rdflib import Graph
from reasoning.gtfs_routing import GtfsRouter
from reasoning.preference_filter import find_pois

g = Graph()
g.parse("../kg/vienna_mobility_kg.ttl", format="turtle")
router = GtfsRouter(date="20260815")
print(f"KG: {len(g)} triples, GTFS router ready")

## 1. Dog-friendly parks — the query that came back empty before

In the original version of this notebook, this exact query (from
Albertinaplatz) returned zero reachable results out of 10 candidates shown —
a real, honestly-reported outcome under direct-only routing, but not a very
useful one. Worth re-running now.

In [ ]:
origin_lon, origin_lat = 16.368378925180206, 48.204590314427854  # Albertinaplatz area

results = find_pois(g, router, origin_lon, origin_lat, poi_classes=["Park"],
                     required_amenities=["Dogs allowed"], top_n=10)
n_reachable = sum(1 for r in results if r["reachable"])
print(f"{n_reachable} of {len(results)} dog-friendly parks reachable within 2 transfers\n")
for r in results:
    status = f"{r['travel_time_min']} min ({r['num_transfers']} transfers)" if r["reachable"] else "no connection"
    print(f"  {r['name']:30} {status}")

## 2. Same category filters as before, now with realistic coverage

Parks near the museum used in the earlier notebooks, and playgrounds with a
trampoline — both were sparse or empty before.

In [ ]:
q = """
PREFIX schema: <https://schema.org/>
PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
SELECT ?lon ?lat WHERE {
    ?m a schema:Museum ; schema:name "Sammlung alter Musikinstrumente" ; geo:long ?lon ; geo:lat ?lat .
}
"""
m_lon, m_lat = [(float(r.lon), float(r.lat)) for r in g.query(q)][0]

parks = find_pois(g, router, m_lon, m_lat, poi_classes=["Park"], top_n=10)
print("Parks, ranked by travel time:")
for r in parks:
    status = f"{r['travel_time_min']} min ({r['num_transfers']} transfers)" if r["reachable"] else "no connection"
    print(f"  {r['name']:30} {status}")

trampolines = find_pois(g, router, m_lon, m_lat, poi_classes=["PlaygroundArea"],
                         required_amenities=["Trampolin"], top_n=10)
n_reach = sum(1 for r in trampolines if r["reachable"])
print(f"\nPlaygrounds with a trampoline: {n_reach} of {len(trampolines)} reachable")
for r in trampolines[:5]:
    status = f"{r['travel_time_min']} min ({r['num_transfers']} transfers)" if r["reachable"] else "no connection"
    print(f"  {r['name']:30} {status}")

## 3. A hard time cutoff, now meaningful with transfers included

30 minutes, any museum, from the same origin.

In [ ]:
museums_30min = find_pois(g, router, origin_lon, origin_lat, poi_classes=["Museum"],
                           max_travel_time_min=30, top_n=15)
print(f"{len(museums_30min)} museums within 30 min:")
for r in museums_30min:
    print(f"  {r['name']:35} {r['travel_time_min']} min ({r['num_transfers']} transfers)")

## Findings

**The mechanism is unchanged, the results are transformed.** Same
`find_pois()` API, same category/amenity SPARQL pattern, same ranking logic —
the only thing that changed is `GtfsRouter` now finding real routes instead
of reporting "no direct connection" for the vast majority of queries. The
dog-friendly-park query that returned zero results in the original version of
this notebook now returns real, ranked, mostly-single-digit-or-low-double-digit-minute
options.

**Performance mattered, not just correctness.** The naive way to add
transfers — call a richer `estimate_travel_time()` per candidate — would have
made `find_pois()` unusably slow for large categories (Parks alone has 1,051
candidates). Restructuring around `reachable_from()` once + `travel_time_to()`
per candidate turned "several minutes per query" into "a few seconds,"
independent of how much richer the underlying search got.

**Open question, still unresolved:** what should the Service Layer do with
the small remaining set of truly unreachable POIs (still possible even with 2
transfers)? Same options as before — fall back to walking-distance-only,
relax the transfer bound further for that one query, or just be upfront that
nothing matched — now a much smaller and more honest edge case than it was
under direct-only routing.